In [5]:
import pandas

pandas.options.mode.copy_on_write = True
pandas.options.display.float_format = '{:,.1f}'.format

test_script = pandas.read_csv('../relational-test-script.csv')
test_script.set_index('filename', inplace=True)

test_script.loc[test_script['abstract_domain'] == 'Box', 'abstract_domain'] = 'Interval'

print('\\def\\testScriptSuites{'+str(test_script[test_script['abstract_domain'] == 'Interval'].index.size)+'\\xspace}')
print('\\def\\testScriptAssertions{'+str(test_script[test_script['abstract_domain'] == 'Interval']['test_cases'].sum())+'\\xspace}')

polka = test_script[test_script['abstract_domain'] == 'Polka'].rename(columns={'passed_cases': 'polyhedra_passed_cases'})['polyhedra_passed_cases']
octagon = test_script[test_script['abstract_domain'] == 'Octagon'].rename(columns={'passed_cases': 'octagon_passed_cases'})['octagon_passed_cases']
interval = test_script[test_script['abstract_domain'] == 'Interval'].rename(columns={'passed_cases': 'interval_passed_cases'})['interval_passed_cases']
test_cases = test_script[test_script['abstract_domain'] == 'Interval']['test_cases']

test_script = pandas.concat([polka, octagon, interval, test_cases], axis=1)


control_flow = test_script.filter(regex='(block|labels|br|if|return|select|switch|unreachable|nop)', axis=0)
functions    = test_script.filter(regex='(call|func|fac|forward)', axis=0)
memory       = test_script.filter(regex='(address|align|endianness|memory|load|store|left-to-right)', axis=0).drop(labels=['float_memory.wast'])
local_vars   = test_script.filter(regex='(local)', axis=0)
global_vars  = test_script.filter(regex='(global)', axis=0)
stack        = test_script.filter(regex='(stack|unwind)', axis=0)
constants    = test_script.filter(regex='(const)', axis=0)
integers     = test_script.filter(regex='(int|i32|i64|traps)', axis=0)
floats       = test_script.filter(regex='(f32|f64|float)', axis=0)
conversions  = test_script.filter(regex='(conversions)', axis=0)
modules      = test_script.filter(regex='(elem|start|exports|imports|linking|names|utf8|custom|table|token|type|data|inline-module|comments|unreached-invalid|binary-leb128)', axis=0)
0
def insert_category(df, id, category):
    df.insert(0, 'category', category)
    df.insert(0, 'category_id', id)
    files = df.index.size
    df = df.groupby(['category_id', 'category']).sum()
    df.insert(0, 'test_suites', files)
    return df

categories = [
    insert_category(constants, 1, 'Constants'),
    insert_category(integers, 2, 'Integers'),
    insert_category(floats, 3, 'Floats'),
    insert_category(conversions, 4, 'Conversions'),
    insert_category(control_flow, 5, 'Control Flow'),
    insert_category(functions, 6, 'Functions'),
    insert_category(stack, 7, 'Stack'),
    insert_category(local_vars, 8, 'Local Vars.'),
    insert_category(global_vars, 9, 'Global Vars.'),
    insert_category(memory, 10, 'Memory'),
    insert_category(modules, 11, 'Modules')
]


test_script = pandas.concat(categories, sort=False)
test_script.insert(1, 'polyhedra_passed_percent', test_script['polyhedra_passed_cases'] / test_script['test_cases'] * 100)
test_script.insert(3, 'octagon_passed_percent', test_script['octagon_passed_cases'] / test_script['test_cases'] * 100)
test_script.insert(5, 'interval_passed_percent', test_script['interval_passed_cases'] / test_script['test_cases'] * 100)


table = pandas.DataFrame({
    'Category': test_script.index.get_level_values(1),
    'Test Files': test_script['test_suites'],
    'Assertions': test_script['test_cases'],
    'Polyhedra': test_script['polyhedra_passed_percent'],
    'Octagons':  test_script['octagon_passed_percent'],
    'Interval':  test_script['interval_passed_percent'],
})
print(table.to_latex(index=False, float_format="{:0.0f}\%".format))


\def\testScriptSuites{71\xspace}
\def\testScriptAssertions{16517\xspace}
\begin{tabular}{lrrrrr}
\toprule
Category & Test Files & Assertions & Polyhedra & Octagons & Interval \\
\midrule
Constants & 1 & 300 & 100\% & 100\% & 100\% \\
Integers & 5 & 909 & 100\% & 100\% & 100\% \\
Floats & 10 & 11897 & 100\% & 100\% & 100\% \\
Conversions & 1 & 593 & 100\% & 100\% & 100\% \\
Control Flow & 11 & 831 & 100\% & 100\% & 100\% \\
Functions & 6 & 321 & 100\% & 100\% & 100\% \\
Stack & 3 & 54 & 100\% & 100\% & 100\% \\
Local Vars. & 3 & 93 & 100\% & 100\% & 100\% \\
Global Vars. & 1 & 48 & 100\% & 100\% & 100\% \\
Memory & 11 & 852 & 100\% & 100\% & 100\% \\
Modules & 20 & 757 & 100\% & 100\% & 100\% \\
\bottomrule
\end{tabular}

